In [1]:
#Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [2]:
# Loading the data 
df = pd.read_csv(r"D:\NTI-ML\nti-projects\employee-attrition-analysis\data\raw\employee_attrition_course.csv")
df.head()  

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,RecordID,ManagerNotes
0,41.0,Yes,Travel_Rarely,1102,sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,NaN,19479,8,Y,Yes,11,3,1,80,0,8.0,0,1,6.0,4,0,5,100001,NaN
1,49.0,No,Travel_Frequently,279,Research and Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,NaN,24907,1,Y,No,23,4,4,80,1,10.0,3,3,10.0,7,1,7,100002,NaN
2,37.0,Yes,Travel_Rarely,1373,Research and Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090.0,2396,6,Y,Yes,15,3,2,80,0,7.0,3,3,0.0,0,0,0,100003,Needs Improvement
3,33.0,No,Travel_Frequently,1392,Research and Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909.0,23159,1,Y,Yes,11,3,3,80,0,8.0,3,3,NaN,7,3,0,100004,Average
4,27.0,No,Travel_Rarely,591,Research and Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468.0,16632,9,Y,No,12,3,4,80,1,6.0,3,3,2.0,2,2,2,100005,NaN


In [ ]:
# Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2) ## To show the percentage
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False) # To arrange them
missing_df


,missing_count,missing_pct
ManagerNotes,897,60.00
MonthlyIncome,120,8.03
YearsAtCompany,75,5.02
TotalWorkingYears,61,4.08
Age,44,2.94
EducationField,29,1.94
JobRole,15,1.00


ManagerNotes: has a very high missing percentage (~60%).

In [4]:
# Duplicate Records

# Check for fully duplicated rows (all columns are identical)
full_dup = df.duplicated().sum()
print("Number of fully duplicated rows:", full_dup)

# Check for duplicates while ignoring RecordID
# (because it is a unique identifier and may differ even if the rest of the data is identical)
dup_ignore_id = df.duplicated(subset=[c for c in df.columns if c != 'RecordID']).sum()
print("Number of duplicated rows (ignoring RecordID only):", dup_ignore_id)

Number of fully duplicated rows: 25
Number of duplicated rows (ignoring RecordID only): 25


In [5]:
print("Number of duplicated EmployeeNumber values:", df['EmployeeNumber'].duplicated().sum())
print("Number of duplicated RecordID values:", df['RecordID'].duplicated().sum())

# Display a sample of duplicated employees to understand the nature of the duplicates
dups_sample = df[df.duplicated(subset='EmployeeNumber', keep=False)].sort_values('EmployeeNumber')
dups_sample[['EmployeeNumber', 'RecordID', 'Age', 'Attrition']].head(10)

Number of duplicated EmployeeNumber values: 25
Number of duplicated RecordID values: 45


,EmployeeNumber,RecordID,Age,Attrition
57,75,100058,35.0,No
1476,75,100058,35.0,No
1494,177,100135,26.0,No
134,177,100135,26.0,No
1471,205,100153,53.0,No
152,205,100153,53.0,No
1489,286,100208,36.0,No
207,286,100208,36.0,No
562,780,100563,33.0,Yes
1490,780,100563,33.0,Yes


In [6]:
# Incorrect Data Types
df.dtypes


Age                         float64
Attrition                    object
BusinessTravel               object
DailyRate                     int64
Department                   object
DistanceFromHome              int64
Education                     int64
EducationField               object
EmployeeCount                 int64
EmployeeNumber                int64
EnvironmentSatisfaction       int64
Gender                       object
HourlyRate                    int64
JobInvolvement                int64
JobLevel                      int64
JobRole                      object
JobSatisfaction               int64
MaritalStatus                object
MonthlyIncome               float64
MonthlyRate                   int64
NumCompaniesWorked            int64
Over18                       object
OverTime                     object
PercentSalaryHike             int64
PerformanceRating             int64
RelationshipSatisfaction      int64
StandardHours                 int64
StockOptionLevel            

In [7]:
# Inconsistent Values
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()


--- Attrition ---
Attrition
No     1256
Yes     239
Name: count, dtype: int64

--- BusinessTravel ---
BusinessTravel
Travel_Rarely        1051
Travel_Frequently     281
Non-Travel            151
Travel Rarely           3
Travel-Rarely           3
TRAVEL_RARELY           3
travel_rarely           3
Name: count, dtype: int64

--- Department ---
Department
Research and Development    973
sales                       456
Human Resourses              66
Name: count, dtype: int64

--- EducationField ---
EducationField
Life Sciences       605
Medical             460
Marketing           160
Technical Degree    128
Other                87
NaN                  29
Human Resources      26
Name: count, dtype: int64

--- Gender ---
Gender
Male        878
Female      591
 Male        19
 Female       7
Name: count, dtype: int64

--- JobRole ---
JobRole
Sales Executive              321
Research Scientist           290
Laboratory Technician        259
Manufacturing Director       146
Healthcare Represen

## Detected Text Data Quality Issues

| Column | Issue |
|---|---|
| `BusinessTravel` | The same category is written in multiple formats: `Travel_Rarely`, `Travel Rarely`, `Travel-Rarely`, `TRAVEL_RARELY`, `travel_rarely` |
| `Department` | Inconsistent capitalization (`sales` instead of `Sales`) and a spelling mistake (`Human Resourses` instead of `Human Resources`) |
| `Gender` | Leading whitespace before values: `' Male'`, `' Female'` |
| `JobRole` | Multiple variations of the same job title: `Sales Ex.`, `Sales Executve`, `sales executive` instead of `Sales Executive` |

In [9]:
# Outliers & Invalid Values

# Check for negative values in all numerical columns
num_cols = df.select_dtypes(include=[np.number]).columns
neg_report = {col: (df[col] < 0).sum() for col in num_cols if (df[col] < 0).sum() > 0}

print("Columns containing negative values:", neg_report if neg_report else "No negative values found")

Columns containing negative values: No negative values found


In [10]:
# 1) Check for employees younger than 18 years old
# (considered unrealistic for a full-time employee)
print("Number of employees younger than 18:", (df['Age'] < 18).sum())

df[df['Age'] < 18][['Age']]

Number of employees younger than 18: 2


,Age
799,16.0
872,17.0


In [11]:
# 2) Check for unusually high MonthlyIncome values
# (potential outliers compared to the overall distribution)
print(df[df['MonthlyIncome'] > 30000][['MonthlyIncome']])

print("\nThird Quartile (Q3) =", df['MonthlyIncome'].quantile(0.75))

      MonthlyIncome
108        500000.0
139        250000.0
273        400000.0
1237       320000.0
1451       180000.0

Third Quartile (Q3) = 8468.5


In [12]:
# 3) Check for potential outliers in DistanceFromHome
# (employees with unusually large commuting distances)
print(df[df['DistanceFromHome'] > 30][['DistanceFromHome']])

      DistanceFromHome
179                150
472                 95
1291               120


In [13]:
# 4) Check for logical inconsistencies:
# YearsAtCompany should not exceed TotalWorkingYears
tmp = df.dropna(subset=['YearsAtCompany', 'TotalWorkingYears'])
logic_bad = tmp[tmp['YearsAtCompany'] > tmp['TotalWorkingYears']]

print(
    "Number of records with logical inconsistencies "
    "(YearsAtCompany > TotalWorkingYears):",
    len(logic_bad)
)

Number of records with logical inconsistencies (YearsAtCompany > TotalWorkingYears): 0


> **Summary:** Potential outliers were detected in `MonthlyIncome` and `DistanceFromHome`, along with implausible employee ages (16 and 17 years). No logical inconsistencies were found between `YearsAtCompany` and `TotalWorkingYears`.

In [14]:
# Constant Features

# Identify constant columns (columns with only one unique value)
constant_cols = [
    col for col in df.columns
    if df[col].nunique(dropna=False) == 1
]

print("Constant columns (only one unique value across all rows):", constant_cols)

# Display the unique value for each constant column
for col in constant_cols:
    print(col, "->", df[col].unique())

Constant columns (only one unique value across all rows): ['EmployeeCount', 'Over18', 'StandardHours']
EmployeeCount -> [1]
Over18 -> ['Y']
StandardHours -> [80]


> **Observation:** `EmployeeCount`, `Over18`, and `StandardHours` are constant (zero-variance) features. As they contain no analytical or predictive information, they should be removed from the dataset.

In [15]:
# High Missing Columns
high_missing = missing_df[missing_df['missing_pct'] > 50]
print(high_missing)


              missing_count  missing_pct
ManagerNotes            897         60.0


> **Observation:** The `ManagerNotes` column contains approximately **60% missing values** and consists of **unstructured free-text** data. Since it is outside the scope of this structured data analysis and has a high proportion of missing values, the column will be removed from the dataset.